In [13]:
%pip install python-dotenv --upgrade --quiet
%pip install langchain --quiet
%pip install langchain-groq --quiet

from dotenv import load_dotenv
load_dotenv()

import os
from langchain_groq import ChatGroq

# Using the GROQ_API_KEY stored in your .env file
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)

print("Groq key loaded:", "GROQ_API_KEY" in os.environ)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Groq key loaded: True


In [14]:
question = "Roger has 5 tennis balls. He buys 2 cans. Each can has 3 balls. How many total?"

prompt_standard = f"Answer this question: {question}"

print(llm.invoke(prompt_standard).content)

To find the total number of tennis balls, we need to add the balls Roger already has to the balls he buys. 

Roger already has 5 tennis balls. 
He buys 2 cans, each with 3 balls. So, he buys 2 * 3 = 6 balls.

Now, let's add the balls he already has to the balls he buys: 
5 (initial balls) + 6 (new balls) = 11

So, Roger now has a total of 11 tennis balls.


In [15]:
prompt_cot = f"Answer this question. Let's think step by step. {question}"
print(llm.invoke(prompt_cot).content)

To find the total number of tennis balls, we need to add the balls Roger already has to the balls he buys.

Roger already has 5 tennis balls.

He buys 2 cans, and each can has 3 balls. So, he buys 2 x 3 = 6 balls.

Now, let's add the balls he already has to the balls he buys: 5 + 6 = 11.

So, Roger now has a total of 11 tennis balls.


Why did CoT work better?
	•	It generated intermediate reasoning tokens.
	•	The model attended to its own steps.
	•	The multiplication (2×3) became visible.
	•	The model “debugged” its own solution.

Final CoT Answer: 11 tennis balls


part3b

In [16]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7   # creative for branching
)

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "How can I get my 5-year-old to eat vegetables?"

prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give me one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

In [18]:
prompt_judge = ChatPromptTemplate.from_template("""
I have three proposed solutions for: '{problem}'

1: {sol1}
2: {sol2}
3: {sol3}

Act as a Child Psychologist.
Pick the most sustainable one (no bribery) and explain why.
""")

In [19]:
tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print(tot_chain.invoke(problem))

As a child psychologist, I would recommend **Solution 2: "Fruit and Veggie Face" on their Plate** as the most sustainable approach to encourage your 5-year-old to eat vegetables. This approach is effective because it:

1. **Is non-bribery-based**: Unlike solutions that involve hiding vegetables in other foods or using treats as rewards, this approach focuses on creating a fun and engaging experience that doesn't rely on bribery.
2. **Promotes creativity and involvement**: By involving your child in the process of creating the face, you're encouraging their creativity, problem-solving skills, and sense of ownership over their meal. This can help build their confidence and motivation to try new foods.
3. **Makes mealtime interactive**: This approach turns mealtime into a game, making it more enjoyable and engaging for your child. Interactive experiences like this can help reduce mealtime stress and make healthy eating more appealing.
4. **Educates about different vegetables**: By using a

Best Solution: “Fruit & Veggie Face Plate”


	•	Encourages fun and exploration
	•	No bribery
	•	Improves fine motor skills
	•	Children try veggies willingly
	•	Sustainable eating habit

This is the correct reasoning output.

part3c

In [20]:
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 1-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi")   | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance")| llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Horror")  | llm | StrOutputParser(),
)

In [21]:
prompt_combine = ChatPromptTemplate.from_template("""
I have three movie ideas about '{topic}':

1. Sci-Fi: {draft_scifi}
2. Romance: {draft_romance}
3. Horror: {draft_horror}

Your task: Combine:
- Sci-Fi tech
- Romance passion
- Horror fear

Write 1 paragraph.
""")

In [22]:
got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print(got_chain.invoke("Time Travel"))

In "Ripple Effect," a brilliant but reclusive physicist, struggling to overcome his recent heartbreak, stumbles upon a mysterious time machine in his late grandfather's attic. As he attempts to adjust the device, he inadvertently sends himself back in time to the 1940s, where he meets a beautiful young woman who becomes the love of his life - a dashing World War II pilot. However, their whirlwind romance is soon disrupted by a series of eerie and inexplicable events as they begin to relive the same fateful night over and over, with each iteration bringing a new and terrifying twist. The physicist, desperate to be with his loved one, must navigate the complexities of altering the past to be with her, but each change sets off a catastrophic ripple effect, threatening the very fabric of time itself, and forcing him to confront the dark consequences of his actions.
